In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from src.degenderizer import BasicDegenderizer, AdvancedDegenderizer
from src.models import DistilBERTClassifier

In [ ]:
DATA_PATH = "data/sentence_sets_trimmed.csv"
LABEL_COLUMN = "applicant_gender"
TEXT_COLUMN = "s1_s2"

DEGENDERIZERS = [
    "src/degenderizer/nouns.txt",
    "src/degenderizer/pronouns.txt",
    "src/degenderizer/titles.txt",
]

In [ ]:
# Load dataset
df = pd.read_csv(DATA_PATH, encoding="ISO-8859-1")
print("Dataset shape:", df.shape)

In [ ]:
# Create degendering pipelines for each degenderizer
basic_pipeline = Pipeline([("basic", BasicDegenderizer(paths=DEGENDERIZERS))])
advanced_pipeline = Pipeline([("advanced", AdvancedDegenderizer(paths=DEGENDERIZERS))])

In [ ]:
# Apply both pipelines to the text column
df["basic_degendered"] = basic_pipeline.transform(df[TEXT_COLUMN].tolist())
df["advanced_degendered"] = advanced_pipeline.transform(df[TEXT_COLUMN].tolist())

print(df[[TEXT_COLUMN, "basic_degendered", "advanced_degendered"]].head())

In [ ]:
# For training, we use advanced
df["degendered_text"] = df["advanced_degendered"]

In [ ]:
# Convert categorical labels to factors / integers
df[LABEL_COLUMN], class_mapping = pd.factorize(df[LABEL_COLUMN])
print("Class mapping:", dict(enumerate(class_mapping)))

In [ ]:
# Train test splits
X_train, X_test, y_train, y_test = train_test_split(
    df["degendered_text"],
    df[LABEL_COLUMN],
    test_size=0.2,
    random_state=42,
    stratify=df[LABEL_COLUMN],
)

print("Train size:", len(X_train), "Test size:", len(X_test))

In [ ]:
# DistilBERT classifier model
model = DistilBERTClassifier(
    model_name="distilbert-base-uncased",
    num_labels=len(class_mapping),
)

In [ ]:
model.train(
    X_train.tolist(),
    y_train.tolist(),
    epochs=3,
    batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    output_dir="./models/distilbert_degendered"
)

In [ ]:
metrics = model.test(X_test.tolist(), y_test.tolist())

print("Evaluation Metrics:", metrics)